# GPT From Scratch — نموذج لغوي من الصفر
## بيانات Wikipedia + Hyperparameters محسّنة

---

**قبل البدء:**
`Runtime → Change runtime type → T4 GPU → Save`

**الترتيب:** شغّل الخلايا من الأعلى للأسفل بالترتيب.

| المرحلة | الوقت المتوقع على T4 |
|---------|---------------------|
| تحميل البيانات | ~3 دقائق |
| Pretraining | ~15-20 دقيقة |
| Fine-Tuning | ~3 دقائق |


## الخطوة 1 — تثبيت المكتبات


In [ ]:
!pip install -q tiktoken matplotlib datasets
print('✓ Libraries installed')


## الخطوة 2 — التحقق من GPU


In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if torch.cuda.is_available():
    print('GPU   :', torch.cuda.get_device_name(0))
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print('Memory:', round(mem, 1), 'GB')
else:
    print('WARNING: No GPU — training will be slow')


## الخطوة 3 — إنشاء هيكل المشروع


In [ ]:
import os
dirs = [
    'src/model', 'src/tokenizer', 'src/training', 'src/evaluation',
    'data/pretrain', 'data/finetune/story_completion', 'data/finetune/poetry',
    'checkpoints/pretrained', 'checkpoints/finetuned',
    'results/plots', 'results/sample_generations'
]
for d in dirs:
    os.makedirs(d, exist_ok=True)
    if d.startswith('src'):
        open(f'{d}/__init__.py', 'w').close()
open('src/__init__.py', 'w').close()
print('✓ Project structure created')


## الخطوة 4 — كتابة ملفات الكود

> شغّل الخلايا الست التالية بالترتيب


In [ ]:
%%writefile src/model/attention.py
"""
Multi-Head Causal Self-Attention — implemented from scratch using PyTorch.
"""

import math
import torch
import torch.nn as nn


class CausalSelfAttention(nn.Module):
    """
    Single head of causal (masked) self-attention.
    Uses a lower-triangular mask so each token can only attend to past tokens.
    """

    def __init__(self, d_model: int, head_size: int, context_length: int, dropout: float = 0.1):
        super().__init__()
        self.head_size = head_size

        self.W_q = nn.Linear(d_model, head_size, bias=False)
        self.W_k = nn.Linear(d_model, head_size, bias=False)
        self.W_v = nn.Linear(d_model, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)

        # Causal mask: lower-triangular boolean tensor
        self.register_buffer(
            "mask",
            torch.tril(torch.ones(context_length, context_length))
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape  # batch, seq_len, d_model

        Q = self.W_q(x)  # (B, T, head_size)
        K = self.W_k(x)
        V = self.W_v(x)

        # Scaled dot-product attention
        scale = 1.0 / math.sqrt(self.head_size)
        attn = Q @ K.transpose(-2, -1) * scale       # (B, T, T)
        attn = attn.masked_fill(self.mask[:T, :T] == 0, float("-inf"))
        attn = torch.softmax(attn, dim=-1)
        attn = self.dropout(attn)

        out = attn @ V  # (B, T, head_size)
        return out


class MultiHeadAttention(nn.Module):
    """
    Multi-Head Causal Self-Attention.
    Concatenates n_heads independent attention heads then projects.
    """

    def __init__(self, d_model: int, n_heads: int, context_length: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        head_size = d_model // n_heads

        self.heads = nn.ModuleList([
            CausalSelfAttention(d_model, head_size, context_length, dropout)
            for _ in range(n_heads)
        ])
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Concatenate outputs of all heads along the last dimension
        out = torch.cat([h(x) for h in self.heads], dim=-1)  # (B, T, d_model)
        out = self.dropout(self.proj(out))
        return out


In [ ]:
%%writefile src/model/transformer.py
"""
GPT Decoder-Only Transformer — implemented entirely from scratch using PyTorch.

Architecture (per block):
  x → LayerNorm → MultiHeadAttention → residual
    → LayerNorm → FeedForward (MLP)  → residual

Final head:
  x → LayerNorm → Linear → logits over vocabulary
"""

import torch
import torch.nn as nn
from .attention import MultiHeadAttention


# ---------------------------------------------------------------------------
# Sub-components
# ---------------------------------------------------------------------------

class LayerNorm(nn.Module):
    """Custom Layer Normalisation (avoids bias=False quirk in older PyTorch)."""

    def __init__(self, d_model: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))
        self.bias = nn.Parameter(torch.zeros(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        return self.weight * x_norm + self.bias


class FeedForward(nn.Module):
    """
    Position-wise Feed-Forward Network.
    Hidden dim is 4× d_model (standard GPT convention).
    Uses GELU activation (same as GPT-2).
    """

    def __init__(self, d_model: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class TransformerBlock(nn.Module):
    """
    Single Transformer decoder block:
        - Pre-norm Multi-Head Causal Self-Attention
        - Pre-norm Feed-Forward MLP
    Each sub-layer uses a residual (skip) connection.
    """

    def __init__(self, config: dict):
        super().__init__()
        d_model = config["emb_dim"]
        self.ln1 = LayerNorm(d_model)
        self.attn = MultiHeadAttention(
            d_model=d_model,
            n_heads=config["n_heads"],
            context_length=config["context_length"],
            dropout=config["drop_rate"],
        )
        self.ln2 = LayerNorm(d_model)
        self.ff = FeedForward(d_model, dropout=config["drop_rate"])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Attention sub-layer with residual
        x = x + self.attn(self.ln1(x))
        # FFN sub-layer with residual
        x = x + self.ff(self.ln2(x))
        return x


# ---------------------------------------------------------------------------
# Full GPT Model
# ---------------------------------------------------------------------------

class GPTModel(nn.Module):
    """
    GPT (Decoder-Only Transformer) built entirely from scratch.

    Config keys:
        vocab_size      – vocabulary size
        context_length  – maximum sequence length
        emb_dim         – embedding / model dimension
        n_heads         – number of attention heads
        n_layers        – number of Transformer blocks
        drop_rate       – dropout probability
    """

    def __init__(self, config: dict):
        super().__init__()
        self.config = config
        vocab_size = config["vocab_size"]
        context_length = config["context_length"]
        d_model = config["emb_dim"]

        # Token + position embeddings
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(context_length, d_model)
        self.emb_drop = nn.Dropout(config["drop_rate"])

        # Stack of Transformer blocks
        self.blocks = nn.Sequential(*[
            TransformerBlock(config) for _ in range(config["n_layers"])
        ])

        # Final layer norm + language-model head
        self.ln_f = LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        # Weight tying (standard GPT trick: lm_head shares weights with tok_emb)
        self.lm_head.weight = self.tok_emb.weight

        # Initialise weights
        self.apply(self._init_weights)

    def _init_weights(self, module: nn.Module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        assert T <= self.config["context_length"], (
            f"Sequence length {T} exceeds context_length {self.config['context_length']}"
        )

        positions = torch.arange(T, device=idx.device)
        x = self.emb_drop(self.tok_emb(idx) + self.pos_emb(positions))
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)  # (B, T, vocab_size)
        return logits

    def num_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters())


# ---------------------------------------------------------------------------
# Text generation helpers
# ---------------------------------------------------------------------------

@torch.no_grad()
def generate(
    model: GPTModel,
    idx: torch.Tensor,
    max_new_tokens: int,
    temperature: float = 1.0,
    top_k: int = None,
) -> torch.Tensor:
    """
    Auto-regressive text generation with optional temperature scaling and top-k sampling.

    Args:
        model: trained GPTModel
        idx: (1, T) tensor of seed token IDs
        max_new_tokens: number of tokens to generate
        temperature: > 1 → more random, < 1 → more deterministic
        top_k: if set, sample from top-k most probable tokens only
    """
    model.eval()
    context_length = model.config["context_length"]

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_length:]          # crop to context window
        logits = model(idx_cond)                     # (1, T, vocab_size)
        logits = logits[:, -1, :] / temperature      # last position

        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float("-inf")

        probs = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)

    return idx


In [ ]:
%%writefile src/tokenizer/bpe_tokenizer.py
"""
BPE Tokenizer wrapper — uses tiktoken (cl100k_base) which natively supports
Arabic and other Unicode scripts via UTF-8 byte-level encoding.

Why cl100k_base?
    - 100,277 token vocabulary (GPT-4 / text-embedding-ada-002).
    - Full UTF-8 coverage → handles Arabic, diacritics, mixed-language text.
    - Larger vocabulary than gpt2 (50,257) reduces sequence length for Arabic.

Arabic-specific handling:
    - Optional diacritic (tashkeel) stripping via `normalize_arabic`.
    - Optional Hamza/Alef normalisation for lower out-of-vocabulary rate.
"""

import re
import unicodedata
import torch
import tiktoken


# ---------------------------------------------------------------------------
# Arabic text normalisation utilities
# ---------------------------------------------------------------------------

# Unicode range for Arabic diacritics (harakat / tashkeel)
_ARABIC_DIACRITICS = re.compile(r"[\u064B-\u065F\u0670]")

# Alef variants → plain Alef
_ALEF_VARIANTS = str.maketrans("أإآٱ", "ااaa".replace("a", "ا"))


def normalize_arabic(text: str, remove_diacritics: bool = True, normalize_alef: bool = True) -> str:
    """
    Light Arabic text normalisation.

    Steps:
        1. Normalise Unicode to NFC (composed form).
        2. Optionally remove tashkeel (diacritics).
        3. Optionally unify Alef variants (أ إ آ → ا).
        4. Collapse multiple whitespace characters.
    """
    text = unicodedata.normalize("NFC", text)
    if remove_diacritics:
        text = _ARABIC_DIACRITICS.sub("", text)
    if normalize_alef:
        text = text.translate(_ALEF_VARIANTS)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# ---------------------------------------------------------------------------
# Tokenizer class
# ---------------------------------------------------------------------------

class BPETokenizer:
    """
    Thin wrapper around tiktoken providing:
        - `encode`  : str → List[int]
        - `decode`  : List[int] / Tensor → str
        - `encode_tensor` : str → (1, T) LongTensor
        - Arabic normalisation hook
        - Special token support (<|endoftext|>)

    Vocabulary size: 100,277 (cl100k_base)
    """

    SPECIAL = {"<|endoftext|>"}

    def __init__(self, normalize_arabic_text: bool = True):
        self._enc = tiktoken.get_encoding("cl100k_base")
        self.normalize_arabic_text = normalize_arabic_text
        self.vocab_size = self._enc.n_vocab  # 100,277

    # ------------------------------------------------------------------
    # Core API
    # ------------------------------------------------------------------

    def encode(self, text: str) -> list[int]:
        if self.normalize_arabic_text:
            text = normalize_arabic(text)
        return self._enc.encode(text, allowed_special=self.SPECIAL)

    def decode(self, token_ids) -> str:
        if isinstance(token_ids, torch.Tensor):
            token_ids = token_ids.squeeze().tolist()
            if isinstance(token_ids, int):
                token_ids = [token_ids]
        return self._enc.decode(token_ids)

    def encode_tensor(self, text: str, device: str = "cpu") -> torch.Tensor:
        """Return shape (1, T) LongTensor ready for model input."""
        ids = self.encode(text)
        return torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

    # ------------------------------------------------------------------
    # Convenience
    # ------------------------------------------------------------------

    @property
    def eot_token(self) -> int:
        return self._enc.eot_token  # end-of-text special token ID

    def __repr__(self) -> str:
        return (
            f"BPETokenizer(encoding='cl100k_base', vocab_size={self.vocab_size}, "
            f"normalize_arabic={self.normalize_arabic_text})"
        )


In [ ]:
%%writefile src/tokenizer/vocab.py
"""
Vocabulary statistics and analysis utilities.
"""

from collections import Counter
from .bpe_tokenizer import BPETokenizer


def compute_vocab_coverage(text: str, tokenizer: BPETokenizer) -> dict:
    """
    Analyse how well the tokenizer covers the given text.

    Returns a dict with:
        total_chars    – number of characters in raw text
        total_tokens   – number of tokens produced
        compression    – chars-per-token ratio (higher = more efficient)
        top_tokens     – 20 most frequent token strings
    """
    token_ids = tokenizer.encode(text)
    total_tokens = len(token_ids)
    total_chars = len(text)
    compression = total_chars / total_tokens if total_tokens else 0

    # Decode each token individually to get string representations
    token_strings = [tokenizer.decode([tid]) for tid in token_ids]
    top_tokens = Counter(token_strings).most_common(20)

    return {
        "total_chars": total_chars,
        "total_tokens": total_tokens,
        "compression_ratio": round(compression, 3),
        "vocab_size": tokenizer.vocab_size,
        "top_20_tokens": top_tokens,
    }


def print_vocab_report(text: str, tokenizer: BPETokenizer, label: str = "Dataset"):
    stats = compute_vocab_coverage(text, tokenizer)
    print(f"\n{'='*50}")
    print(f"  Vocabulary Report — {label}")
    print(f"{'='*50}")
    print(f"  Characters      : {stats['total_chars']:,}")
    print(f"  Tokens          : {stats['total_tokens']:,}")
    print(f"  Compression     : {stats['compression_ratio']:.2f} chars/token")
    print(f"  Vocab size      : {stats['vocab_size']:,}  (cl100k_base BPE)")
    print(f"\n  Top 20 tokens:")
    for tok, cnt in stats["top_20_tokens"]:
        display = repr(tok)
        print(f"    {display:<20}  {cnt:>6}×")
    print(f"{'='*50}\n")


In [ ]:
%%writefile src/training/pretrain.py
"""
Phase 1 — Pretraining Script
==============================
Trains the GPT model on raw text data using next-token prediction (causal LM).

Usage:
    python -m src.training.pretrain

Outputs:
    checkpoints/pretrained/model_best.pth   – best checkpoint (lowest val loss)
    results/plots/pretrain_loss.png         – loss curves
    results/sample_generations/pretrain_*.txt
"""

import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", ".."))

import json
import math
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

from src.model.transformer import GPTModel, generate
from src.tokenizer.bpe_tokenizer import BPETokenizer
from src.tokenizer.vocab import print_vocab_report

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

GPT_CONFIG = {
    "vocab_size"     : 100_277,
    "context_length" : 256,
    "emb_dim"        : 256,
    "n_heads"        : 8,
    "n_layers"       : 6,
    "drop_rate"      : 0.0,   # 0.0 works better with limited data
}

TRAIN_CONFIG = {
    "batch_size"   : 8,
    "num_epochs"   : 15,
    "lr"           : 5e-4,
    "weight_decay" : 0.1,
    "eval_freq"    : 10,
    "eval_iter"    : 5,
    "warmup_steps" : 50,
    "grad_clip"    : 1.0,
}

DATA_PATH   = Path("data/pretrain/data.txt")
CKPT_DIR    = Path("checkpoints/pretrained")
PLOTS_DIR   = Path("results/plots")
GEN_DIR     = Path("results/sample_generations")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
GEN_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------------
# Dataset
# ---------------------------------------------------------------------------

class TextDataset(Dataset):
    """Sliding-window dataset for causal language modelling."""

    def __init__(self, token_ids: list[int], context_length: int, stride: int):
        self.examples = []
        for start in range(0, len(token_ids) - context_length, stride):
            x = token_ids[start : start + context_length]
            y = token_ids[start + 1 : start + context_length + 1]
            self.examples.append((torch.tensor(x), torch.tensor(y)))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


def build_dataloaders(text: str, tokenizer: BPETokenizer, config: dict):
    token_ids = tokenizer.encode(text)
    ctx = config["context_length"]

    split = int(0.9 * len(token_ids))
    train_ids, val_ids = token_ids[:split], token_ids[split:]

    train_ds = TextDataset(train_ids, ctx, stride=ctx)
    val_ds   = TextDataset(val_ids,   ctx, stride=ctx)

    train_loader = DataLoader(train_ds, batch_size=TRAIN_CONFIG["batch_size"],
                              shuffle=True,  drop_last=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=TRAIN_CONFIG["batch_size"],
                              shuffle=False, drop_last=False, num_workers=0)
    return train_loader, val_loader


# ---------------------------------------------------------------------------
# Loss helpers
# ---------------------------------------------------------------------------

def loss_batch(x, y, model, device):
    x, y = x.to(device), y.to(device)
    logits = model(x)                              # (B, T, V)
    return nn.functional.cross_entropy(logits.flatten(0, 1), y.flatten())


def loss_loader(loader, model, device, n_batches=None):
    if len(loader) == 0:
        return float("nan")
    n = min(n_batches or len(loader), len(loader))
    total = sum(
        loss_batch(x, y, model, device).item()
        for i, (x, y) in enumerate(loader) if i < n
    )
    return total / n


# ---------------------------------------------------------------------------
# LR scheduler — linear warmup then cosine decay
# ---------------------------------------------------------------------------

def get_lr(step: int, warmup: int, total_steps: int, base_lr: float) -> float:
    if step < warmup:
        return base_lr * (step + 1) / warmup
    progress = (step - warmup) / max(1, total_steps - warmup)
    return base_lr * (0.5 * (1 + math.cos(math.pi * progress)))


# ---------------------------------------------------------------------------
# Training loop
# ---------------------------------------------------------------------------

def train(model, train_loader, val_loader, optimizer, device, tokenizer):
    cfg = TRAIN_CONFIG
    total_steps = len(train_loader) * cfg["num_epochs"]

    train_losses, val_losses, tokens_seen_log = [], [], []
    tokens_seen = 0
    global_step = 0
    best_val = float("inf")

    for epoch in range(cfg["num_epochs"]):
        model.train()
        for x, y in train_loader:
            # LR warmup / cosine decay
            lr = get_lr(global_step, cfg["warmup_steps"], total_steps, cfg["lr"])
            for pg in optimizer.param_groups:
                pg["lr"] = lr

            optimizer.zero_grad()
            loss = loss_batch(x, y, model, device)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            optimizer.step()

            tokens_seen += x.numel()
            global_step += 1

            if global_step % cfg["eval_freq"] == 0:
                model.eval()
                with torch.no_grad():
                    tl = loss_loader(train_loader, model, device, cfg["eval_iter"])
                    vl = loss_loader(val_loader,   model, device, cfg["eval_iter"])
                model.train()
                train_losses.append(tl)
                val_losses.append(vl)
                tokens_seen_log.append(tokens_seen)
                print(
                    f"Ep {epoch+1:02d} | Step {global_step:05d} | lr={lr:.2e} | "
                    f"Train {tl:.3f} | Val {vl:.3f}"
                )

                if vl < best_val:
                    best_val = vl
                    torch.save(model.state_dict(), CKPT_DIR / "model_best.pth")

        # End-of-epoch sample
        print(f"\n--- Sample after epoch {epoch+1} ---")
        _print_sample(model, tokenizer, device)
        print("-" * 70)

    # Save final checkpoint + optimizer
    torch.save(
        {"model_state_dict": model.state_dict(),
         "optimizer_state_dict": optimizer.state_dict(),
         "config": GPT_CONFIG},
        CKPT_DIR / "model_final.pth",
    )
    return train_losses, val_losses, tokens_seen_log


# ---------------------------------------------------------------------------
# Sample generation helper
# ---------------------------------------------------------------------------

START_PROMPTS = [
    "في البداية كان",
    "Once upon a time",
    "The model learned to",
]


def _print_sample(model, tokenizer, device, prompt=None):
    model.eval()
    prompt = prompt or START_PROMPTS[0]
    idx = tokenizer.encode_tensor(prompt, device=str(device))
    out = generate(model, idx, max_new_tokens=60, temperature=0.8, top_k=40)
    text = tokenizer.decode(out[0])
    print(text.replace("\n", " "))
    model.train()
    return text


def save_samples(model, tokenizer, device, tag="pretrain"):
    model.eval()
    results = []
    for i, prompt in enumerate(START_PROMPTS):
        idx = tokenizer.encode_tensor(prompt, device=str(device))
        out = generate(model, idx, max_new_tokens=100, temperature=0.8, top_k=40)
        text = tokenizer.decode(out[0])
        results.append({"prompt": prompt, "generation": text})
        path = GEN_DIR / f"{tag}_sample_{i+1}.txt"
        path.write_text(text, encoding="utf-8")
    model.train()
    print(f"\nSaved {len(results)} generation samples to {GEN_DIR}/")


# ---------------------------------------------------------------------------
# Plot
# ---------------------------------------------------------------------------

def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
    fig, ax1 = plt.subplots(figsize=(8, 5))
    ax1.plot(epochs_seen, train_losses, label="Training loss")
    ax1.plot(epochs_seen, val_losses, linestyle="-.", label="Validation loss")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Cross-Entropy Loss")
    ax1.legend(loc="upper right")
    ax1.xaxis.set_major_locator(MaxNLocator(integer=True))

    ax2 = ax1.twiny()
    ax2.plot(tokens_seen, train_losses, alpha=0)
    ax2.set_xlabel("Tokens seen")

    fig.tight_layout()
    out_path = PLOTS_DIR / "pretrain_loss.png"
    plt.savefig(out_path, dpi=150)
    print(f"Loss plot saved → {out_path}")
    plt.close()


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

def main():
    device = (
        torch.device("cuda") if torch.cuda.is_available()
        else torch.device("mps") if hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
        else torch.device("cpu")
    )
    print(f"Device: {device}")

    # ── Tokenizer ─────────────────────────────────────────────────────────
    tokenizer = BPETokenizer(normalize_arabic_text=True)
    print(tokenizer)

    # ── Data ──────────────────────────────────────────────────────────────
    text = DATA_PATH.read_text(encoding="utf-8")
    print(f"Dataset: {len(text):,} characters")
    print_vocab_report(text, tokenizer, label="Pretrain Data")

    train_loader, val_loader = build_dataloaders(text, tokenizer, GPT_CONFIG)
    print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

    # ── Model ─────────────────────────────────────────────────────────────
    torch.manual_seed(42)
    model = GPTModel(GPT_CONFIG).to(device)
    print(f"Parameters: {model.num_parameters():,}")

    # ── Optimizer ─────────────────────────────────────────────────────────
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=TRAIN_CONFIG["lr"],
        weight_decay=TRAIN_CONFIG["weight_decay"],
    )

    # ── Baseline loss ─────────────────────────────────────────────────────
    with torch.no_grad():
        init_train = loss_loader(train_loader, model, device, 5)
        init_val   = loss_loader(val_loader,   model, device, 5)
    print(f"\nInitial  Train loss: {init_train:.3f}  |  Val loss: {init_val:.3f}")
    print(f"Initial  Perplexity: {math.exp(init_train):.1f}")

    # ── Train ──────────────────────────────────────────────────────────────
    train_losses, val_losses, tokens_seen = train(
        model, train_loader, val_loader, optimizer, device, tokenizer
    )

    # ── Plots & samples ───────────────────────────────────────────────────
    num_epochs = TRAIN_CONFIG["num_epochs"]
    epochs_tensor = torch.linspace(0, num_epochs, len(train_losses)).tolist()
    plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)
    save_samples(model, tokenizer, device, tag="pretrain")

    final_val = val_losses[-1] if val_losses else float("nan")
    print(f"\nFinal Val loss: {final_val:.3f}  |  Perplexity: {math.exp(final_val):.1f}")
    print("Pretraining complete.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/training/finetune.py
"""
Phase 2 — Supervised Fine-Tuning (SFT)
=======================================
Fine-tunes the pretrained GPT on instruction-following data.

Data format  (JSON list or JSONL):
    {"instruction": "...", "input": "...", "output": "..."}

Prompt template:
    ### Instruction:
    {instruction}

    ### Input:
    {input}

    ### Response:
    {output}<|endoftext|>

Usage:
    python -m src.training.finetune --data data/finetune/story_completion/sft_data.json
"""

import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", ".."))

import json
import math
import argparse
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

from src.model.transformer import GPTModel, generate
from src.tokenizer.bpe_tokenizer import BPETokenizer

# ---------------------------------------------------------------------------
# Paths & config
# ---------------------------------------------------------------------------

PRETRAIN_CKPT = Path("checkpoints/pretrained/model_best.pth")
FINETUNE_DIR  = Path("checkpoints/finetuned")
PLOTS_DIR     = Path("results/plots")
GEN_DIR       = Path("results/sample_generations")
FINETUNE_DIR.mkdir(parents=True, exist_ok=True)

GPT_CONFIG = {
    "vocab_size"     : 100_277,
    "context_length" : 256,
    "emb_dim"        : 256,
    "n_heads"        : 8,
    "n_layers"       : 6,
    "drop_rate"      : 0.2,   # higher dropout to reduce overfitting
}

SFT_CONFIG = {
    "batch_size"   : 4,
    "num_epochs"   : 3,
    "lr"           : 5e-5,
    "weight_decay" : 0.1,
    "eval_freq"    : 5,
    "eval_iter"    : 3,
    "grad_clip"    : 1.0,
}


# ---------------------------------------------------------------------------
# Prompt formatting
# ---------------------------------------------------------------------------

PROMPT_TEMPLATE = (
    "### Instruction:\n{instruction}\n\n"
    "### Input:\n{input}\n\n"
    "### Response:\n{output}"
)


def format_prompt(record: dict, include_output: bool = True) -> str:
    """Format a SFT record into the prompt template."""
    output = record.get("output", "") if include_output else ""
    return PROMPT_TEMPLATE.format(
        instruction=record.get("instruction", ""),
        input=record.get("input", ""),
        output=output,
    )


# ---------------------------------------------------------------------------
# Dataset
# ---------------------------------------------------------------------------

class SFTDataset(Dataset):
    """
    Each example is one full formatted prompt (instruction + response).
    Loss is computed on all tokens (standard SFT).
    Sequences are padded / truncated to context_length.
    """

    def __init__(self, records: list[dict], tokenizer: BPETokenizer, context_length: int):
        self.examples = []
        self.context_length = context_length
        eot = tokenizer.eot_token

        for rec in records:
            text = format_prompt(rec) + tokenizer.decode([eot])
            ids = tokenizer.encode(text)

            # Truncate or pad to exactly context_length+1
            if len(ids) < context_length + 1:
                ids = ids + [eot] * (context_length + 1 - len(ids))
            else:
                ids = ids[:context_length + 1]

            x = torch.tensor(ids[:-1], dtype=torch.long)
            y = torch.tensor(ids[1:],  dtype=torch.long)
            self.examples.append((x, y))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


def load_sft_data(path: Path) -> list[dict]:
    text = path.read_text(encoding="utf-8").strip()
    if path.suffix == ".jsonl":
        return [json.loads(line) for line in text.splitlines() if line.strip()]
    return json.loads(text)


def build_sft_loaders(records, tokenizer, config):
    split = int(0.9 * len(records))
    train_recs, val_recs = records[:split], records[split:]

    train_ds = SFTDataset(train_recs, tokenizer, config["context_length"])
    val_ds   = SFTDataset(val_recs,   tokenizer, config["context_length"])

    train_loader = DataLoader(train_ds, batch_size=SFT_CONFIG["batch_size"],
                              shuffle=True,  drop_last=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=SFT_CONFIG["batch_size"],
                              shuffle=False, drop_last=False, num_workers=0)
    return train_loader, val_loader


# ---------------------------------------------------------------------------
# Loss helpers
# ---------------------------------------------------------------------------

def loss_batch(x, y, model, device):
    x, y = x.to(device), y.to(device)
    logits = model(x)
    return nn.functional.cross_entropy(logits.flatten(0, 1), y.flatten())


def loss_loader(loader, model, device, n=None):
    if len(loader) == 0:
        return float("nan")
    n = min(n or len(loader), len(loader))
    total = sum(
        loss_batch(x, y, model, device).item()
        for i, (x, y) in enumerate(loader) if i < n
    )
    return total / n


# ---------------------------------------------------------------------------
# Training loop
# ---------------------------------------------------------------------------

def train_sft(model, train_loader, val_loader, optimizer, device, tokenizer, records):
    cfg = SFT_CONFIG
    train_losses, val_losses = [], []
    best_val = float("inf")
    global_step = 0

    for epoch in range(cfg["num_epochs"]):
        model.train()
        for x, y in train_loader:
            optimizer.zero_grad()
            loss = loss_batch(x, y, model, device)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            optimizer.step()
            global_step += 1

            if global_step % cfg["eval_freq"] == 0:
                model.eval()
                with torch.no_grad():
                    tl = loss_loader(train_loader, model, device, cfg["eval_iter"])
                    vl = loss_loader(val_loader,   model, device, cfg["eval_iter"])
                model.train()
                train_losses.append(tl)
                val_losses.append(vl)
                print(
                    f"Ep {epoch+1:02d} | Step {global_step:04d} | "
                    f"Train {tl:.3f} | Val {vl:.3f}"
                )
                if vl < best_val:
                    best_val = vl
                    torch.save(model.state_dict(), FINETUNE_DIR / "model_sft_best.pth")

        print(f"\n--- Sample after epoch {epoch+1} ---")
        _demo_sample(model, tokenizer, device, records)
        print("-" * 70)

    torch.save(
        {"model_state_dict": model.state_dict(),
         "config": GPT_CONFIG},
        FINETUNE_DIR / "model_sft_final.pth",
    )
    return train_losses, val_losses


# ---------------------------------------------------------------------------
# Demo generation
# ---------------------------------------------------------------------------

def _demo_sample(model, tokenizer, device, records, n=2):
    model.eval()
    for rec in records[:n]:
        prompt = format_prompt(rec, include_output=False) + "\n"
        idx = tokenizer.encode_tensor(prompt, device=str(device))
        out = generate(model, idx, max_new_tokens=80, temperature=0.7, top_k=40)
        text = tokenizer.decode(out[0])
        print(f"Instruction: {rec['instruction']}")
        print(f"Generated  : {text[len(prompt):].strip()}\n")
    model.train()


def save_finetuned_samples(model, tokenizer, device, records, tag="sft"):
    model.eval()
    results = []
    for i, rec in enumerate(records[:5]):
        prompt = format_prompt(rec, include_output=False) + "\n"
        idx = tokenizer.encode_tensor(prompt, device=str(device))
        out = generate(model, idx, max_new_tokens=120, temperature=0.7, top_k=40)
        generated = tokenizer.decode(out[0])
        results.append({
            "instruction": rec["instruction"],
            "input": rec.get("input", ""),
            "expected": rec.get("output", ""),
            "generated": generated,
        })
        path = GEN_DIR / f"{tag}_sample_{i+1}.txt"
        path.write_text(
            f"=== Instruction ===\n{rec['instruction']}\n\n"
            f"=== Expected ===\n{rec.get('output','')}\n\n"
            f"=== Generated ===\n{generated}",
            encoding="utf-8",
        )
    model.train()
    print(f"Saved {len(results)} SFT samples to {GEN_DIR}/")
    return results


# ---------------------------------------------------------------------------
# Plot
# ---------------------------------------------------------------------------

def plot_sft_losses(train_losses, val_losses, tag="sft"):
    steps = list(range(1, len(train_losses) + 1))
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(steps, train_losses, label="Training loss")
    ax.plot(steps, val_losses,   linestyle="-.", label="Validation loss")
    ax.set_xlabel("Evaluation steps")
    ax.set_ylabel("Cross-Entropy Loss")
    ax.set_title("SFT Loss Curves")
    ax.legend()
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    fig.tight_layout()
    out = PLOTS_DIR / f"{tag}_loss.png"
    plt.savefig(out, dpi=150)
    print(f"SFT loss plot saved → {out}")
    plt.close()


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--data",
        default="data/finetune/story_completion/sft_data.json",
        help="Path to SFT data (JSON list or JSONL)",
    )
    args = parser.parse_args()

    device = (
        torch.device("cuda") if torch.cuda.is_available()
        else torch.device("mps") if hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
        else torch.device("cpu")
    )
    print(f"Device: {device}")

    tokenizer = BPETokenizer(normalize_arabic_text=True)

    # Load SFT data
    records = load_sft_data(Path(args.data))
    print(f"SFT records: {len(records)}")

    train_loader, val_loader = build_sft_loaders(records, tokenizer, GPT_CONFIG)
    print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

    # Load pretrained model
    model = GPTModel(GPT_CONFIG).to(device)
    if PRETRAIN_CKPT.exists():
        state = torch.load(PRETRAIN_CKPT, map_location=device, weights_only=True)
        model.load_state_dict(state)
        print(f"Loaded pretrained weights from {PRETRAIN_CKPT}")
    else:
        print("WARNING: pretrained checkpoint not found, fine-tuning from random init.")

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=SFT_CONFIG["lr"],
        weight_decay=SFT_CONFIG["weight_decay"],
    )

    train_losses, val_losses = train_sft(
        model, train_loader, val_loader, optimizer, device, tokenizer, records
    )

    tag = Path(args.data).parent.name
    plot_sft_losses(train_losses, val_losses, tag=tag)
    save_finetuned_samples(model, tokenizer, device, records, tag=tag)
    print("Fine-tuning complete.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/evaluation/metrics.py
"""
Evaluation Metrics
==================
Implements:
  - Perplexity (PPL)   — measures how well the model predicts held-out text
  - BLEU score         — n-gram overlap for generation quality
  - Repetition rate    — ratio of repeated n-grams (failure mode)
  - Vocabulary richness (TTR) — type-token ratio

Usage:
    from src.evaluation.metrics import compute_perplexity, evaluate_generation
"""

import math
import re
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import DataLoader


# ---------------------------------------------------------------------------
# Perplexity
# ---------------------------------------------------------------------------

def compute_perplexity(
    model: nn.Module,
    data_loader: DataLoader,
    device: torch.device,
    max_batches: int = None,
) -> float:
    """
    Compute perplexity on a DataLoader.

    Perplexity = exp(mean cross-entropy loss)

    Lower is better. A random model over 100k vocab has PPL ≈ 100,277.
    """
    model.eval()
    total_loss = 0.0
    n = 0

    with torch.no_grad():
        for i, (x, y) in enumerate(data_loader):
            if max_batches and i >= max_batches:
                break
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = nn.functional.cross_entropy(logits.flatten(0, 1), y.flatten())
            total_loss += loss.item()
            n += 1

    avg_loss = total_loss / n if n else float("nan")
    return math.exp(avg_loss)


def compute_perplexity_from_text(
    model: nn.Module,
    tokenizer,
    text: str,
    context_length: int,
    device: torch.device,
) -> float:
    """
    Compute perplexity directly from raw text (no DataLoader needed).
    Slides a window of `context_length` tokens across the text.
    """
    model.eval()
    token_ids = tokenizer.encode(text)
    total_loss, count = 0.0, 0

    with torch.no_grad():
        for start in range(0, len(token_ids) - context_length, context_length):
            x = torch.tensor(token_ids[start : start + context_length]).unsqueeze(0).to(device)
            y = torch.tensor(token_ids[start + 1 : start + context_length + 1]).unsqueeze(0).to(device)
            logits = model(x)
            loss = nn.functional.cross_entropy(logits.flatten(0, 1), y.flatten())
            total_loss += loss.item()
            count += 1

    avg_loss = total_loss / count if count else float("nan")
    return math.exp(avg_loss)


# ---------------------------------------------------------------------------
# BLEU score (1-gram to 4-gram, brevity penalty)
# ---------------------------------------------------------------------------

def _ngrams(tokens: list, n: int) -> Counter:
    return Counter(tuple(tokens[i : i + n]) for i in range(len(tokens) - n + 1))


def bleu_score(hypothesis: str, reference: str, max_n: int = 4) -> float:
    """
    Corpus BLEU-n (simplified single-sentence version).
    Returns a score in [0, 1].
    """
    hyp_tokens = hypothesis.split()
    ref_tokens = reference.split()

    if not hyp_tokens:
        return 0.0

    score = 0.0
    for n in range(1, max_n + 1):
        hyp_ng = _ngrams(hyp_tokens, n)
        ref_ng = _ngrams(ref_tokens, n)
        matches = sum(min(hyp_ng[ng], ref_ng[ng]) for ng in hyp_ng)
        total = max(1, sum(hyp_ng.values()))
        precision = matches / total
        score += math.log(precision + 1e-10)

    score = math.exp(score / max_n)

    # Brevity penalty
    bp = 1.0 if len(hyp_tokens) >= len(ref_tokens) else math.exp(
        1 - len(ref_tokens) / max(1, len(hyp_tokens))
    )
    return bp * score


# ---------------------------------------------------------------------------
# Repetition rate
# ---------------------------------------------------------------------------

def repetition_rate(text: str, n: int = 3) -> float:
    """
    Fraction of n-grams that are repeated.
    High repetition → model stuck in loops (failure mode).
    """
    tokens = text.split()
    if len(tokens) < n:
        return 0.0
    ngrams = [tuple(tokens[i : i + n]) for i in range(len(tokens) - n + 1)]
    counts = Counter(ngrams)
    repeated = sum(v - 1 for v in counts.values() if v > 1)
    return repeated / len(ngrams)


# ---------------------------------------------------------------------------
# Type-Token Ratio (vocabulary richness)
# ---------------------------------------------------------------------------

def type_token_ratio(text: str) -> float:
    """
    TTR = unique_words / total_words
    Higher → richer vocabulary.
    """
    tokens = re.findall(r"\w+", text.lower())
    if not tokens:
        return 0.0
    return len(set(tokens)) / len(tokens)


# ---------------------------------------------------------------------------
# Full generation evaluation report
# ---------------------------------------------------------------------------

def evaluate_generation(hypothesis: str, reference: str = None, label: str = "") -> dict:
    """
    Run all metrics on a generated text and return a dict.
    """
    result = {
        "label": label,
        "length_words": len(hypothesis.split()),
        "repetition_rate_3gram": round(repetition_rate(hypothesis, n=3), 4),
        "type_token_ratio": round(type_token_ratio(hypothesis), 4),
    }
    if reference:
        result["bleu"] = round(bleu_score(hypothesis, reference), 4)
    return result


def print_metrics_report(results: list[dict]):
    print("\n" + "=" * 60)
    print("  Evaluation Metrics Report")
    print("=" * 60)
    for r in results:
        print(f"\n  [{r['label']}]")
        print(f"    Words          : {r['length_words']}")
        print(f"    Repetition(3g) : {r['repetition_rate_3gram']:.4f}  (0=none, 1=all repeated)")
        print(f"    TTR            : {r['type_token_ratio']:.4f}  (higher=richer vocab)")
        if "bleu" in r:
            print(f"    BLEU           : {r['bleu']:.4f}")
    print("=" * 60 + "\n")


In [ ]:
%%writefile src/evaluation/error_analysis.py
"""
Error Analysis & Failure Mode Detection
=========================================
Identifies and reports common failure modes in generated text:
  1. Repetitive loops       — same phrase repeated
  2. Context loss           — prompt topic absent from generation
  3. Short / truncated text — model stops too early
  4. Language switch        — Arabic prompt → English output (or vice versa)
  5. Instruction ignoring   — response unrelated to instruction

Usage:
    python -m src.evaluation.error_analysis
"""

import re
import json
from pathlib import Path
from collections import Counter

from src.evaluation.metrics import repetition_rate, type_token_ratio, bleu_score


# ---------------------------------------------------------------------------
# Failure mode detectors
# ---------------------------------------------------------------------------

def _has_arabic(text: str) -> bool:
    return bool(re.search(r"[\u0600-\u06FF]", text))


def _has_english(text: str) -> bool:
    return bool(re.search(r"[a-zA-Z]{3,}", text))


def detect_failures(prompt: str, generation: str, expected: str = None) -> dict:
    """
    Analyse a single generation for failure modes.

    Returns a dict with keys:
        repetition       : repetition rate (0–1)
        ttr              : type-token ratio (0–1)
        is_short         : True if < 10 words
        context_loss     : True if no word from prompt appears in generation
        lang_switch      : True if detected language differs between prompt & gen
        bleu             : BLEU vs expected (if provided)
        failure_modes    : list of detected failure mode strings
    """
    failures = []
    rep = repetition_rate(generation, n=3)
    ttr = type_token_ratio(generation)
    gen_words = generation.split()
    prompt_words = set(re.findall(r"\w+", prompt.lower()))

    # 1. Repetition
    if rep > 0.4:
        failures.append("HIGH_REPETITION")

    # 2. Short output
    is_short = len(gen_words) < 10
    if is_short:
        failures.append("TOO_SHORT")

    # 3. Context loss — none of the non-trivial prompt words found in gen
    content_words = {w for w in prompt_words if len(w) > 3}
    gen_lower = generation.lower()
    context_overlap = sum(1 for w in content_words if w in gen_lower)
    context_loss = len(content_words) > 0 and context_overlap == 0
    if context_loss:
        failures.append("CONTEXT_LOSS")

    # 4. Language switch
    prompt_arabic = _has_arabic(prompt)
    gen_arabic = _has_arabic(generation)
    gen_english = _has_english(generation)
    lang_switch = (prompt_arabic and not gen_arabic and gen_english) or \
                  (not prompt_arabic and gen_arabic)
    if lang_switch:
        failures.append("LANGUAGE_SWITCH")

    result = {
        "repetition": round(rep, 4),
        "ttr": round(ttr, 4),
        "is_short": is_short,
        "context_loss": context_loss,
        "lang_switch": lang_switch,
        "failure_modes": failures,
    }
    if expected:
        result["bleu"] = round(bleu_score(generation, expected), 4)

    return result


# ---------------------------------------------------------------------------
# Batch analysis
# ---------------------------------------------------------------------------

def analyze_samples(samples: list[dict]) -> list[dict]:
    """
    Analyse a list of {"prompt", "generation", "expected"(optional)} dicts.
    Returns list of analysis results.
    """
    results = []
    for i, s in enumerate(samples):
        analysis = detect_failures(
            prompt=s.get("prompt", s.get("instruction", "")),
            generation=s.get("generation", ""),
            expected=s.get("expected", s.get("output", None)),
        )
        analysis["sample_id"] = i + 1
        analysis["prompt_preview"] = s.get("prompt", s.get("instruction", ""))[:60]
        results.append(analysis)
    return results


def print_error_report(results: list[dict]):
    print("\n" + "=" * 70)
    print("  Error Analysis & Failure Mode Report")
    print("=" * 70)

    all_failures = []
    for r in results:
        all_failures.extend(r["failure_modes"])
        status = "✓" if not r["failure_modes"] else "✗"
        modes = ", ".join(r["failure_modes"]) if r["failure_modes"] else "none"
        print(f"\n  Sample {r['sample_id']} [{status}]")
        print(f"    Prompt   : {r['prompt_preview']}…")
        print(f"    Rep rate : {r['repetition']:.3f} | TTR: {r['ttr']:.3f} | Short: {r['is_short']}")
        if "bleu" in r:
            print(f"    BLEU     : {r['bleu']:.4f}")
        print(f"    Failures : {modes}")

    print("\n  --- Failure Mode Summary ---")
    counts = Counter(all_failures)
    if counts:
        for mode, cnt in counts.most_common():
            print(f"    {mode:<25} {cnt:>3}× / {len(results)} samples "
                  f"({100*cnt/len(results):.0f}%)")
    else:
        print("    No failures detected.")
    print("=" * 70 + "\n")


# ---------------------------------------------------------------------------
# Load & run from saved generation files
# ---------------------------------------------------------------------------

def run_from_directory(gen_dir: Path, tag: str = ""):
    samples = []
    for path in sorted(gen_dir.glob(f"{tag}*.txt")):
        text = path.read_text(encoding="utf-8")
        # Parse simple file format written by training scripts
        parts = text.split("=== Generated ===\n")
        if len(parts) == 2:
            header, generation = parts
            instruction = ""
            expected = ""
            for line in header.splitlines():
                if line.startswith("=== Instruction ==="):
                    pass
                elif line.startswith("=== Expected ==="):
                    pass
                else:
                    instruction += line + " "
            samples.append({
                "prompt": instruction.strip(),
                "generation": generation.strip(),
            })
        else:
            samples.append({"prompt": path.stem, "generation": text})

    if not samples:
        print(f"No generation files found in {gen_dir} with tag '{tag}'.")
        return

    results = analyze_samples(samples)
    print_error_report(results)
    return results


if __name__ == "__main__":
    gen_dir = Path("results/sample_generations")
    print("--- Pretrain generations ---")
    run_from_directory(gen_dir, tag="pretrain")
    print("\n--- SFT generations ---")
    run_from_directory(gen_dir, tag="sft")


In [ ]:
print('✓ All source files written')


## الخطوة 5 — كتابة بيانات Fine-Tuning (220 مثالاً مدمجة)


In [ ]:
import base64

sft_b64    = '...'
poetry_b64 = '...'

with open('data/finetune/story_completion/sft_data.json', 'wb') as f:
    f.write(base64.b64decode(sft_b64))
with open('data/finetune/poetry/sft_data.json', 'wb') as f:
    f.write(base64.b64decode(poetry_b64))

import json
n = len(json.load(open('data/finetune/story_completion/sft_data.json', encoding='utf-8')))
print(f'✓ SFT data ready — {n} instruction pairs')


## الخطوة 6 — تحميل بيانات Wikipedia (المصدر الحقيقي)

> هذه الخطوة تستغرق ~3 دقائق — **لا تتخطاها**

> ستحمّل ~15 مليون حرف من Wikipedia العربية والإنجليزية


In [ ]:
import re
from datasets import load_dataset
import base64

def clean_wiki(text):
    text = re.sub(r'={2,}[^=]*={2,}', '', text)
    text = re.sub(r'\[\d+\]', '', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'https?://\S+', '', text)
    return text.strip()

print('Loading Arabic Wikipedia (3000 articles)...')
ds_ar = load_dataset('wikimedia/wikipedia', '20231101.ar', split='train[:3000]')
articles_ar = [clean_wiki(r['text']) for r in ds_ar if len(r['text']) > 300]
text_ar = '\n\n'.join(articles_ar)
print(f'Arabic : {len(articles_ar)} articles | {len(text_ar):,} chars')

print('Loading English Wikipedia (500 articles)...')
ds_en = load_dataset('wikimedia/wikipedia', '20231101.en', split='train[:500]')
articles_en = [clean_wiki(r['text']) for r in ds_en if len(r['text']) > 300]
text_en = '\n\n'.join(articles_en)
print(f'English: {len(articles_en)} articles | {len(text_en):,} chars')

ORIGINAL_B64 = '...'
original = base64.b64decode(ORIGINAL_B64).decode('utf-8')
combined = original + '\n\n' + text_ar + '\n\n' + text_en

with open('data/pretrain/data.txt', 'w', encoding='utf-8') as f:
    f.write(combined)

print()
print('='*50)
print(f'Original data : {len(original):>12,} chars')
print(f'Wikipedia AR  : {len(text_ar):>12,} chars')
print(f'Wikipedia EN  : {len(text_en):>12,} chars')
print(f'Total         : {len(combined):>12,} chars')
print('='*50)
print('Data ready for training')


## الخطوة 7 — التدريب الأولي (Pretraining)

**الإعدادات المحسّنة:**
- `drop_rate = 0.0` — أفضل مع البيانات المحدودة
- `num_epochs = 15` — تدريب أطول
- `lr = 5e-4` — معدل تعلم مرتفع قليلاً
- `batch_size = 8` — يستفيد من GPU

> وقت التدريب على T4: **~15-20 دقيقة**


In [ ]:
!cd /content && python -m src.training.pretrain


## الخطوة 8 — نتائج التدريب الأولي


In [ ]:
import math
from IPython.display import Image, display

display(Image('results/plots/pretrain_loss.png'))

# عرض النصوص المُولَّدة بعد التدريب
import os
gen_dir = 'results/sample_generations'
pretrain_files = [f for f in sorted(os.listdir(gen_dir)) if f.startswith('pretrain')]
print('\n' + '='*60)
print('  نماذج توليد النص بعد Pretraining')
print('='*60)
for fname in pretrain_files:
    print(f'\n--- {fname} ---')
    print(open(f'{gen_dir}/{fname}', encoding='utf-8').read())


## الخطوة 9 — التعديل الدقيق (Fine-Tuning)

**الإعدادات المحسّنة لتقليل Overfitting:**
- `num_epochs = 3` — تقليل Epochs
- `lr = 5e-5` — معدل تعلم أبطأ
- `drop_rate = 0.2` — Dropout أعلى
- `weight_decay = 0.1` — تنظيم أقوى

> وقت التدريب على T4: **~3 دقائق**


In [ ]:
!cd /content && python -m src.training.finetune --data data/finetune/story_completion/sft_data.json


## الخطوة 10 — نتائج التعديل الدقيق


In [ ]:
from IPython.display import Image, display
import os

display(Image('results/plots/story_completion_loss.png'))

gen_dir = 'results/sample_generations'
sft_files = [f for f in sorted(os.listdir(gen_dir)) if 'story' in f or 'sft' in f]
print('\n' + '='*60)
print('  مقارنة قبل وبعد Fine-Tuning')
print('='*60)
for fname in sft_files:
    print(f'\n--- {fname} ---')
    print(open(f'{gen_dir}/{fname}', encoding='utf-8').read())


## الخطوة 11 — التقييم الشامل

**يحسب:** Perplexity | BLEU | Repetition Rate | Error Analysis


In [ ]:
import sys, math, torch
sys.path.insert(0, '/content')

from src.model.transformer import GPTModel, generate
from src.tokenizer.bpe_tokenizer import BPETokenizer
from src.evaluation.metrics import compute_perplexity_from_text, evaluate_generation, print_metrics_report
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = BPETokenizer()

GPT_CONFIG = {
    'vocab_size': 100_277, 'context_length': 256,
    'emb_dim': 256, 'n_heads': 8, 'n_layers': 6, 'drop_rate': 0.0
}

# تحميل النماذج
pretrain_model = GPTModel(GPT_CONFIG).to(device)
pretrain_model.load_state_dict(
    torch.load('checkpoints/pretrained/model_best.pth', map_location=device, weights_only=True))
pretrain_model.eval()

sft_model = GPTModel(GPT_CONFIG).to(device)
sft_model.load_state_dict(
    torch.load('checkpoints/finetuned/model_sft_best.pth', map_location=device, weights_only=True))
sft_model.eval()

# حساب Perplexity
val_text = open('data/pretrain/data.txt', encoding='utf-8').read()[-5000:]
ppl = compute_perplexity_from_text(pretrain_model, tokenizer, val_text, 256, device)

print('='*55)
print('  نتائج التقييم النهائية')
print('='*55)
print(f'  Perplexity (Pretrained) : {ppl:.1f}')
print(f'  Val Loss (Pretrained)   : {math.log(ppl):.3f}')

# مقاييس النصوص المُولَّدة
import os
gen_dir = 'results/sample_generations'
results = []
for fname in sorted(os.listdir(gen_dir)):
    text = open(f'{gen_dir}/{fname}', encoding='utf-8').read()
    parts = text.split('=== Generated ===\n')
    gen = parts[-1].strip() if len(parts) > 1 else text
    r = evaluate_generation(gen, label=fname.replace('.txt',''))
    results.append(r)

print_metrics_report(results)
print('='*55)


## الخطوة 12 — تحليل الأخطاء


In [ ]:
!cd /content && python -m src.evaluation.error_analysis


## الخطوة 13 — توليد نص تفاعلي

غيّر `prompt` و `temperature` وشغّل الخلية.


In [ ]:
import sys, torch
sys.path.insert(0, '/content')
from src.model.transformer import GPTModel, generate
from src.tokenizer.bpe_tokenizer import BPETokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = BPETokenizer()

GPT_CONFIG = {
    'vocab_size': 100_277, 'context_length': 256,
    'emb_dim': 256, 'n_heads': 8, 'n_layers': 6, 'drop_rate': 0.0
}

model = GPTModel(GPT_CONFIG).to(device)
model.load_state_dict(
    torch.load('checkpoints/pretrained/model_best.pth',
               map_location=device, weights_only=True))
model.eval()

def gen(prompt, tokens=120, temperature=0.8, top_k=40):
    idx = tokenizer.encode_tensor(prompt, device=str(device))
    out = generate(model, idx, max_new_tokens=tokens,
                   temperature=temperature, top_k=top_k)
    return tokenizer.decode(out[0])

# ── غيّر هنا ──────────────────────────────────────────
prompts = [
    'في البداية كان',
    'العلم نور والجهل ظلام',
    'Once upon a time',
]
temperature = 0.8   # 0.5=محافظ | 0.8=متوازن | 1.2=إبداعي
# ──────────────────────────────────────────────────────

for p in prompts:
    print(f'Prompt: {p}')
    print(gen(p, temperature=temperature))
    print('-'*60)


## الخطوة 14 — ملخص النتائج للجنة


In [ ]:
import math, os, torch
from pathlib import Path

print()
print('='*60)
print('   ملخص المشروع النهائي — GPT From Scratch')
print('='*60)

# بيانات
data = open('data/pretrain/data.txt', encoding='utf-8').read()
print(f'  بيانات التدريب : {len(data):>12,} حرف')

# نموذج
print(f'  معاملات النموذج: {30_470_912:>12,}')
print(f'  مفردات Tokenizer: {100_277:>11,} توكن')

# checkpoints
ckpt = Path('checkpoints/pretrained/model_best.pth')
if ckpt.exists():
    size = ckpt.stat().st_size / 1e6
    print(f'  حجم النموذج   : {size:>11.1f} MB')

print()
print('  النتائج:')
print(f'  Perplexity قبل التدريب : ~105,000')
print(f'  Perplexity بعد التدريب : مشاهدة في الخطوة 11')
print(f'  SFT val loss           : مشاهدة في الخطوة 10')

gen_files = os.listdir('results/sample_generations')
print(f'  ملفات التوليد          : {len(gen_files)} ملف')
print('='*60)


## الخطوة 15 — حفظ النتائج على Google Drive

> **مهم:** Colab يحذف كل شيء عند إغلاق الجلسة — احفظ الآن


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
dest = '/content/drive/MyDrive/GPT_From_Scratch_Results'
os.makedirs(dest, exist_ok=True)

shutil.copytree('checkpoints', f'{dest}/checkpoints', dirs_exist_ok=True)
shutil.copytree('results',     f'{dest}/results',     dirs_exist_ok=True)
shutil.copy('data/pretrain/data.txt', f'{dest}/data_combined.txt')

total = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, files in os.walk(dest)
    for f in files
) / 1e6
print(f'✓ Saved {total:.0f} MB to {dest}')
